# CreditWise AI — Dataset Exploration and EDA

This notebook examines the loan-risk data before any cleaning or modeling. The goal is to understand its structure, quality, target balance, and observable relationships that may inform later analysis.

**Important:** All findings below are calculated from the original CSV. This notebook does not modify, remove, or impute any observations and does not train a model. Group differences are descriptive associations, not evidence of causation. Always consider sample size and business context when interpreting them.

## 1. Load and inspect the dataset

The path is relative to the project root and also works when the notebook kernel starts in `notebooks/`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titleweight"] = "bold"
pd.options.display.float_format = "{:,.3f}".format

candidate_paths = [
    Path("data/raw/credit_risk_dataset.csv"),
    Path("../data/raw/credit_risk_dataset.csv"),
]
data_path = next((path for path in candidate_paths if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError(
        "Could not find data/raw/credit_risk_dataset.csv from the project root "
        "or notebooks directory."
    )

df = pd.read_csv(data_path)
print(f"Loaded: {data_path.resolve()}")
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head(10))

In [ ]:
print("Column names:")
display(pd.Series(df.columns, name="column"))

print("Data types:")
display(df.dtypes.rename("dtype").to_frame())

print("Dataset information:")
df.info()

print("Descriptive statistics (numeric and categorical):")
display(df.describe(include="all").T)

## 2. Data quality assessment

The checks below report potential issues without changing the original data. Age above 100 is treated as an unusual-value screening threshold for review, not proof that a record is erroneous. Employment longer than age is also flagged for review.

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100),
}).sort_values("missing_count", ascending=False)
print(f"Exact duplicate rows: {df.duplicated().sum():,}")
print("Missing values by column:")
display(missing_summary.style.format({"missing_count": "{:,.0f}", "missing_percent": "{:.2f}%"}))

numeric_columns = df.select_dtypes(include="number").columns.tolist()
categorical_columns = df.select_dtypes(exclude="number").columns.tolist()
print("Numerical columns:", numeric_columns)
print("Categorical columns:", categorical_columns)

quality_checks = pd.Series({
    "Negative income": (df["person_income"] < 0).sum(),
    "Negative employment length": (df["person_emp_length"] < 0).sum(),
    "Age below 0": (df["person_age"] < 0).sum(),
    "Age above 100 (unusual; review)": (df["person_age"] > 100).sum(),
    "Employment length greater than age": (df["person_emp_length"] > df["person_age"]).sum(),
    "Non-positive loan amount": (df["loan_amnt"] <= 0).sum(),
    "Loan status outside {0, 1}": (~df["loan_status"].isin([0, 1])).sum(),
    "Loan-to-income ratio outside [0, 1]": (~df["loan_percent_income"].between(0, 1)).sum(),
}, name="records_flagged")
print("Potentially invalid or unusual values (counts):")
display(quality_checks.to_frame().style.format("{:,.0f}"))

if df["loan_status"].isna().any():
    print("Note: loan_status contains missing values; class percentages below use non-missing labels.")
if df["loan_status"].dropna().isin([0, 1]).all():
    print("Target check: observed non-missing loan_status values are 0 and/or 1 only.")
else:
    print("Target check: unexpected or missing target values require review before modeling.")

## 3. Target variable analysis: `loan_status`

`0` means non-default and `1` means default. The percentage is computed among records with a known target. Class imbalance means one class is substantially more represented; the chart and figures below provide the dataset-specific evidence.

In [ ]:
target_labels = {0: "Non-default (0)", 1: "Default (1)"}
target_counts = df["loan_status"].value_counts(dropna=True).reindex([0, 1], fill_value=0)
target_percentages = target_counts.div(target_counts.sum()).mul(100)
target_summary = pd.DataFrame({
    "class": [target_labels[value] for value in target_counts.index],
    "count": target_counts.to_numpy(),
    "percent": target_percentages.to_numpy(),
})
display(target_summary.style.format({"count": "{:,.0f}", "percent": "{:.2f}%"}))

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(target_summary["class"], target_summary["count"], color=["#4C78A8", "#F58518"])
ax.set_title("Loan Status Class Distribution")
ax.set_xlabel("Loan outcome")
ax.set_ylabel("Number of borrowers")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:,.0f}"))
for bar, count, percent in zip(bars, target_summary["count"], target_summary["percent"]):
    ax.annotate(f"{count:,.0f}\n{percent:.2f}%", (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 4), textcoords="offset points", ha="center", va="bottom")
ax.set_ylim(0, max(target_summary["count"].max() * 1.15, 1))
plt.tight_layout()
plt.show()

if target_percentages.empty or target_percentages.isna().all():
    print("Class imbalance cannot be assessed because there are no observed target values.")
elif len(target_percentages) < 2:
    print("Only one target class is present among known labels; this is a severe class-coverage issue.")
else:
    minority_share = target_percentages.min()
    majority_share = target_percentages.max()
    print(f"Observed default rate: {target_percentages.loc[1]:.2f}%.")
    print(f"Minority class share is {minority_share:.2f}% versus {majority_share:.2f}% for the majority class.")
    print("This is an imbalanced target; later evaluation should include class-sensitive metrics and an appropriate validation strategy.")

## 4. Exploratory visualizations

### Numerical feature distributions

The income histogram uses a logarithmic horizontal axis to make the long upper tail easier to inspect; the underlying values are unchanged.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(data=df, x="person_age", bins=30, color="#4C78A8", ax=axes[0, 0])
axes[0, 0].set(title="Borrower Age Distribution", xlabel="Age (years)", ylabel="Borrowers")

sns.histplot(data=df, x="person_income", bins=40, color="#59A14F", ax=axes[0, 1])
axes[0, 1].set_xscale("log")
axes[0, 1].set(title="Annual Income Distribution (log x-axis)", xlabel="Annual income (currency units; log scale)", ylabel="Borrowers")

sns.histplot(data=df, x="loan_amnt", bins=35, color="#F28E2B", ax=axes[1, 0])
axes[1, 0].set(title="Loan Amount Distribution", xlabel="Loan amount (currency units)", ylabel="Loans")
axes[1, 0].xaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:,.0f}"))

sns.histplot(data=df, x="loan_int_rate", bins=35, color="#B07AA1", ax=axes[1, 1])
axes[1, 1].set(title="Loan Interest Rate Distribution", xlabel="Interest rate (%)", ylabel="Loans")

for ax in axes.flat:
    ax.title.set_fontsize(12)
plt.tight_layout()
plt.show()

### Categorical feature distributions

Bars show the observed record counts in each category. Category labels are retained as they appear in the CSV.

In [ ]:
category_specs = [
    ("loan_grade", "Loan Grade Distribution", ["A", "B", "C", "D", "E", "F", "G"]),
    ("loan_intent", "Loan Purpose Distribution", None),
    ("person_home_ownership", "Home Ownership Distribution", None),
]
fig, axes = plt.subplots(1, 3, figsize=(19, 6))
for ax, (column, title, order) in zip(axes, category_specs):
    counts = df[column].value_counts(dropna=False)
    if order is None:
        order = counts.index.tolist()
    sns.countplot(data=df, x=column, order=order, color="#4C78A8", ax=ax)
    ax.set_title(title)
    ax.set_xlabel(column.replace("_", " ").title())
    ax.set_ylabel("Borrowers")
    ax.tick_params(axis="x", rotation=35)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:,.0f}"))
plt.tight_layout()
plt.show()

### Default rates across categories

Each rate is the mean of the binary `loan_status` within the group. Bar labels include the number of records contributing to each rate. These comparisons are descriptive and should not be read as causal effects.

In [ ]:
def grouped_default_rate(data, column, order=None):
    """Return group size and observed default rate without altering the input data."""
    summary = data.groupby(column, dropna=False)["loan_status"].agg(
        borrowers="count", default_rate="mean"
    )
    if order is not None:
        summary = summary.reindex(order)
    else:
        summary = summary.sort_values("default_rate", ascending=False)
    return summary.dropna(subset=["default_rate"])

def plot_default_rates(ax, summary, title, xlabel):
    labels = [str(value) for value in summary.index]
    rates = summary["default_rate"].mul(100)
    bars = ax.bar(labels, rates, color="#E15759")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Observed default rate (%)")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0f}%"))
    ax.tick_params(axis="x", rotation=35)
    for bar, rate, count in zip(bars, rates, summary["borrowers"]):
        ax.annotate(f"{rate:.1f}%\nn={count:,.0f}",
                    (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=8)
    ax.set_ylim(0, max(float(rates.max()) * 1.25, 1))

grade_rates = grouped_default_rate(df, "loan_grade", order=["A", "B", "C", "D", "E", "F", "G"])
purpose_rates = grouped_default_rate(df, "loan_intent")
ownership_rates = grouped_default_rate(df, "person_home_ownership")

fig, axes = plt.subplots(1, 3, figsize=(19, 7))
plot_default_rates(axes[0], grade_rates, "Default Rate by Loan Grade", "Loan grade")
plot_default_rates(axes[1], purpose_rates, "Default Rate by Loan Purpose", "Loan purpose")
plot_default_rates(axes[2], ownership_rates, "Default Rate by Home Ownership", "Home ownership")
plt.tight_layout()
plt.show()

### Default rate by income group and loan amount relative to income

Income quartiles are calculated from observed incomes. The scatter plot compares income with requested loan amount and overlays reference lines for loan amounts equal to 10%, 25%, and 50% of annual income. Log scaling is used for income because its observed range is wide.

In [ ]:
df["income_group"] = pd.qcut(df["person_income"], q=4, duplicates="drop")
income_rates = grouped_default_rate(df, "income_group")
income_rates = income_rates.sort_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_default_rates(axes[0], income_rates, "Default Rate by Annual Income Quartile", "Annual income quartile")
axes[0].tick_params(axis="x", rotation=25)

status_palette = {0: "#4C78A8", 1: "#E15759"}
sns.scatterplot(data=df, x="person_income", y="loan_amnt", hue="loan_status",
                hue_order=[0, 1], palette=status_palette, alpha=0.22, s=18,
                linewidth=0, ax=axes[1], rasterized=True)
axes[1].set_xscale("log")
axes[1].set_title("Loan Amount Relative to Annual Income")
axes[1].set_xlabel("Annual income (currency units; log scale)")
axes[1].set_ylabel("Loan amount (currency units)")
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:,.0f}"))
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:,.0f}"))
income_line = np.geomspace(df["person_income"].min(), df["person_income"].max(), 200)
for ratio in (0.10, 0.25, 0.50):
    axes[1].plot(income_line, income_line * ratio, linestyle="--", linewidth=1.2,
                 label=f"Loan = {ratio:.0%} of income", color="#333333", alpha=0.8)
axes[1].legend(title="Loan status / reference", fontsize=8)
plt.tight_layout()
plt.show()

print("Income-group default rates and sample sizes:")
display(income_rates.assign(default_percent=income_rates["default_rate"].mul(100))
        [["borrowers", "default_percent"]]
        .style.format({"borrowers": "{:,.0f}", "default_percent": "{:.2f}%"}))

## 5. Business insights and modeling considerations

The five observations below are computed from this dataset at runtime. They describe patterns for further investigation; they do not establish that a feature causes default. Review category sample sizes, data quality, fairness, and possible leakage before using any feature in a decision-support system.

In [ ]:
def rate_description(summary, group_name):
    """Format the observed highest/lowest group rates and their sample counts."""
    valid = summary.dropna(subset=["default_rate"])
    if valid.empty:
        return f"Default rates by {group_name} could not be calculated from observed labels."
    high_key = valid["default_rate"].idxmax()
    low_key = valid["default_rate"].idxmin()
    high = valid.loc[high_key]
    low = valid.loc[low_key]
    return (
        f"Among observed {group_name} groups, {high_key} has the highest default rate "
        f"({high['default_rate']:.1%}, n={high['borrowers']:,.0f}) and {low_key} the lowest "
        f"({low['default_rate']:.1%}, n={low['borrowers']:,.0f})."
    )

default_rate = df["loan_status"].mean()
observed_target_count = int(df["loan_status"].notna().sum())
grade_insight = rate_description(grade_rates, "loan grade")
purpose_insight = rate_description(purpose_rates, "loan purpose")
ownership_insight = rate_description(ownership_rates, "home ownership")
income_insight = rate_description(income_rates, "income quartile")
missing_columns = missing_summary[missing_summary["missing_count"] > 0]
missing_text = ", ".join(
    f"{column}: {int(row['missing_count']):,} ({row['missing_percent']:.2f}%)"
    for column, row in missing_columns.iterrows()
) or "none"

observations = [
    f"The observed default rate is {default_rate:.2%} across {observed_target_count:,} records with a known target.",
    grade_insight,
    income_insight,
    purpose_insight,
    ownership_insight,
]
print("Five calculated observations:")
for number, observation in enumerate(observations, start=1):
    print(f"{number}. {observation}")

print("\nPotential credit-risk indicators to investigate (descriptive, not causal):")
print(f"- Grade, income group, loan purpose, and home-ownership differences summarized above.")
print(f"- Loan amount relative to income is directly represented by loan_percent_income; its observed correlation with loan_status is {df['loan_percent_income'].corr(df['loan_status']):.3f} (Pearson correlation).")
print("- Treat these as candidate signals to validate, not standalone approval rules.")

print("\nData quality issues requiring review/treatment before modeling:")
print(f"- Missing values: {missing_text}.")
print(f"- Exact duplicate records: {df.duplicated().sum():,}; review duplicates before deciding how they should be handled.")
print(f"- Ages above 100: {(df['person_age'] > 100).sum():,}; investigate these unusual records, rather than automatically deleting them.")
print(f"- Employment length greater than age: {(df['person_emp_length'] > df['person_age']).sum():,}; review plausibility and units.")
print(f"- Negative income: {(df['person_income'] < 0).sum():,}; negative employment length: {(df['person_emp_length'] < 0).sum():,}.")

print("\nInitial recommendations for the machine-learning phase:")
print("- Establish a documented train/validation/test strategy and a simple baseline before selecting a model.")
print("- Decide, with domain guidance, how to handle missing values, duplicates, and implausible values; fit any preprocessing on training data only.")
print("- Account for the observed target imbalance with suitable metrics (for example precision, recall, F1, PR-AUC, and calibration), not accuracy alone.")
print("- Check leakage, subgroup performance, fairness, interpretability, and decision thresholds before presenting outputs as lending recommendations.")
print("- No records have been cleaned or removed, and no model has been trained in this notebook.")